# AgriRakshak cloud training

Trains a MobileNetV3-Small classifier for bell pepper, potato, tomato, and an out-of-scope plant class. Images come from the PlantVillage color dataset. The workflow preserves its leaf-grouped test split, creates a leaf-grouped validation split, evaluates confidence, and exports the exact ONNX bundle required by the API.

**Important:** PlantVillage uses controlled backgrounds. Final exhibition claims require separate testing on real phone/field photographs.

In [ ]:
import subprocess, sys
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout or 'No NVIDIA GPU detected. In Colab choose Runtime > Change runtime type > T4 GPU.')
if result.returncode != 0:
    raise RuntimeError('A GPU runtime is required for the full training run.')

In [ ]:
!git clone --depth 1 --branch codex/cloud-training-notebook https://github.com/kaalakhatta/AGRIRAKSHAK.git
%cd AGRIRAKSHAK/ml
!python -m pip install -q -e '.[cloud]'

In [ ]:
from pathlib import Path
WORK = Path('/content/agrirakshak-training')
DATASET = WORK / 'dataset'
RUN = WORK / 'baseline-v1'
BUNDLE = WORK / 'bundle'
WORK.mkdir(parents=True, exist_ok=True)
print(WORK)

In [ ]:
!agrirakshak-prepare-plantvillage --output {DATASET} --max-per-class 1600 --unsupported-per-source-class 120
!cat {DATASET / 'split-manifest.summary.json'}

In [ ]:
# One-epoch smoke test proves the complete data and GPU training path first.
!agrirakshak-train --manifest {DATASET / 'split-manifest.jsonl'} --data-root {DATASET} --output {WORK / 'smoke'} --epochs 1 --batch-size 32 --workers 2 --freeze-features --class-balanced

In [ ]:
# Final baseline. Expect this cell to take tens of minutes on a free T4 GPU.
!agrirakshak-train --manifest {DATASET / 'split-manifest.jsonl'} --data-root {DATASET} --output {RUN} --epochs 15 --batch-size 64 --workers 2 --class-balanced

In [ ]:
!agrirakshak-evaluate --manifest {DATASET / 'split-manifest.jsonl'} --data-root {DATASET} --checkpoint {RUN / 'best.pt'} --output {RUN / 'evaluation'} --batch-size 64 --workers 2
!agrirakshak-export --checkpoint {RUN / 'best.pt'} --evaluation {RUN / 'evaluation' / 'metrics.json'} --output {BUNDLE} --version baseline-v1
!ls -lh {BUNDLE}

In [ ]:
import json
metrics = json.loads((RUN / 'evaluation' / 'metrics.json').read_text())
print('Test macro F1:', metrics['test']['macro_f1'])
print('Confidence threshold:', metrics['confidence_policy']['threshold'])
print('Coverage:', metrics['confidence_policy']['coverage'])
print('Review per-class metrics and the confusion matrix before deploying.')

In [ ]:
import shutil
from google.colab import files
archive = shutil.make_archive('/content/agrirakshak-baseline-v1', 'zip', BUNDLE)
files.download(archive)
# Also download evidence needed for review.
evidence = shutil.make_archive('/content/agrirakshak-evaluation', 'zip', RUN / 'evaluation')
files.download(evidence)